In [1]:
import sys
sys.path.insert(0, '..')

import yaml
import torch
import torch.nn as nn

from src.neural_networks import CifarMLP
from src.quantizer import DeadZoneLDZCompander
from src.utils_quantization import attach_weight_quantizers, toggle_quantization, UniformSymmetric
from src.datasets import build_dataset_cifar
from src.train import train, validate
from src.structured_loss import LOSS_REGISTRY
from src.bobs_calculator import compare_model

QUANTIZER_REGISTRY = {
    "uniform": UniformSymmetric,
    "deadzone": DeadZoneLDZCompander,
}

In [ ]:
CONFIG_PATH = "../configs/deadzone_mlp.yaml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Override for notebook use
cfg["device"] = "cpu"
cfg["workers"] = 0

# Seed for reproducibility
seed = cfg.get("seed", 42)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

DEVICE = torch.device(cfg["device"])
print(f"Config: {CONFIG_PATH}")
print(f"Device: {DEVICE}, Seed: {seed}")
cfg

In [3]:
# Build model
model = CifarMLP()

# Attach quantizer if configured
q_cfg = cfg.get("quantizer")
if q_cfg:
    q_cls = QUANTIZER_REGISTRY[q_cfg["name"]]
    attach_weight_quantizers(
        model,
        exclude_layers=q_cfg.get("exclude_layers", []),
        quantizer=q_cls,
        quantizer_kwargs=q_cfg.get("kwargs", {}),
        enabled=True,
    )
    toggle_quantization(model, enabled=True)

model.to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters())}")
model

Attached weight quantizer to layer: net.0
Attached weight quantizer to layer: net.2
Attached weight quantizer to layer: net.4
Parameters: 379780


CifarMLP(
  (net): Sequential(
    (0): ParametrizedLinear(
      in_features=3072, out_features=120, bias=True
      (parametrizations): ModuleDict(
        (weight): ParametrizationList(
          (0): FakeQuantParametrization(
            (quantizer): DeadZoneLDZCompander()
          )
        )
      )
    )
    (1): ReLU()
    (2): ParametrizedLinear(
      in_features=120, out_features=84, bias=True
      (parametrizations): ModuleDict(
        (weight): ParametrizationList(
          (0): FakeQuantParametrization(
            (quantizer): DeadZoneLDZCompander()
          )
        )
      )
    )
    (3): ReLU()
    (4): ParametrizedLinear(
      in_features=84, out_features=10, bias=True
      (parametrizations): ModuleDict(
        (weight): ParametrizationList(
          (0): FakeQuantParametrization(
            (quantizer): DeadZoneLDZCompander()
          )
        )
      )
    )
  )
)

In [4]:
# Build optimizer (3-way param group split: base / dz / bit)
from run_training import build_optimizer, build_loss_fns

optimizer = build_optimizer(model, cfg)
loss_fns = build_loss_fns(cfg)
criterion = nn.CrossEntropyLoss().to(DEVICE)

epochs = 10
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

print(f"Optimizer: {cfg['optimizer'].get('type', 'adamw')} ({len(optimizer.param_groups)} groups)")
print(f"Loss terms: {[t['name'] for t in cfg.get('loss_terms', [])] or 'CE only'}")
print(f"Epochs: {epochs}")

Optimizer: adamw (3 groups)
Loss terms: CE only
Epochs: 10


/Users/mikkeldahl/CoDeQ/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Data
train_loader, val_loader = build_dataset_cifar(
    workers=cfg.get("workers", 0),
    batch_size=cfg.get("batch_size", 128),
    img_size=cfg.get("img_size", 32),
)

# Optionally use a subset: set cfg["data_subset"] to a fraction, e.g. 0.1 for 10%
subset_frac = cfg.get("data_subset")
if subset_frac and subset_frac < 1.0:
    from torch.utils.data import Subset
    import random
    for name, loader in [("train", train_loader), ("val", val_loader)]:
        n = len(loader.dataset)
        k = max(1, int(n * subset_frac))
        indices = random.sample(range(n), k)
        subset = Subset(loader.dataset, indices)
        new_loader = torch.utils.data.DataLoader(
            subset,
            batch_size=loader.batch_size,
            shuffle=(name == "train"),
            num_workers=loader.num_workers,
            pin_memory=loader.pin_memory,
        )
        if name == "train":
            train_loader = new_loader
        else:
            val_loader = new_loader

print(f"Train: {len(train_loader.dataset)} samples ({len(train_loader)} batches)")
print(f"Val:   {len(val_loader.dataset)} samples ({len(val_loader)} batches)")

/Users/mikkeldahl/CoDeQ/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train: 50000 samples (391 batches)
Val:   10000 samples (79 batches)


In [ ]:
# Training loop
best_prec1 = 0
eval_bobs_every = 2

for epoch in range(epochs):
    lr = optimizer.param_groups[0]["lr"]
    loss, train_acc = train(train_loader, model, criterion, optimizer, epoch, DEVICE, loss_fns=loss_fns)
    lr_scheduler.step()
    val_acc = validate(val_loader, model, criterion, DEVICE)

    is_best = val_acc > best_prec1
    best_prec1 = max(val_acc, best_prec1)
    print(f"Epoch {epoch+1}/{epochs} — lr={lr:.5e}  loss={loss:.4f}  train={train_acc:.2f}  val={val_acc:.2f}{'  [BEST]' if is_best else ''}")

    if cfg.get("eval_bobs", False) and (epoch + 1) % eval_bobs_every == 0:
        bobs = compare_model(
            model,
            default_weight_bitrate=cfg.get("default_weight_bitrate", 8),
            default_activation_bitrate=cfg.get("default_activation_bitrate", 8),
        )
        bobs.print()

Epoch: [0][0/391]	Time 0.059 (0.059)	Data 0.046 (0.046)	Loss 2.3224 (2.3224)	Prec@1 7.812 (7.812)


/Users/mikkeldahl/CoDeQ/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch: [0][50/391]	Time 0.036 (0.035)	Data 0.024 (0.024)	Loss 2.1156 (2.1665)	Prec@1 23.438 (19.914)
Epoch: [0][100/391]	Time 0.034 (0.035)	Data 0.023 (0.024)	Loss 2.0355 (2.1248)	Prec@1 24.219 (22.308)
Epoch: [0][150/391]	Time 0.034 (0.034)	Data 0.023 (0.024)	Loss 1.9618 (2.0965)	Prec@1 28.125 (23.432)
Epoch: [0][200/391]	Time 0.032 (0.034)	Data 0.023 (0.024)	Loss 1.8512 (2.0758)	Prec@1 25.781 (24.312)
Epoch: [0][250/391]	Time 0.032 (0.034)	Data 0.023 (0.024)	Loss 1.9414 (2.0549)	Prec@1 28.125 (25.096)
Epoch: [0][300/391]	Time 0.034 (0.034)	Data 0.024 (0.024)	Loss 1.8679 (2.0371)	Prec@1 28.906 (25.812)
Epoch: [0][350/391]	Time 0.033 (0.034)	Data 0.024 (0.024)	Loss 1.9853 (2.0236)	Prec@1 31.250 (26.378)
Test: [0/79]	Time 0.012 (0.012)	Loss 1.7370 (1.7370)	Prec@1 43.750 (43.750)
Test: [50/79]	Time 0.011 (0.012)	Loss 1.8017 (1.7703)	Prec@1 34.375 (37.040)
 * Prec@1 36.910
Epoch 1/10 — lr=1.00000e-03  loss=2.0138  train=26.77  val=36.91  [BEST]
Epoch: [1][0/391]	Time 0.035 (0.035)	Data 0.